In [6]:
# Core Imports and Environment Setup
# Automatically installs missing dependencies if needed

import sys
import os
import subprocess
from pathlib import Path
import time

# Rosetta Suite Docking
Using Rosetta for advanced peptide-protein docking and structure refinement

In [7]:
# HPC Offloading Functions
# Use PowerShell SSH workaround to submit Rosetta jobs to HPC cluster

def transfer_file_to_hpc_via_powershell(local_file, remote_path, host="hpc.cqls_v2", use_hpc_files=False):
    """
    Transfer a file to HPC cluster using HTTP server + wget or hpc-files (recommended by HPC docs).
    For small files, uses base64 encoding through SSH (most reliable).
    For large files, can use hpc-files.cqls.oregonstate.edu (recommended for large transfers).
    
    Parameters:
    - local_file: Path to local file to transfer
    - remote_path: Destination path on remote server
    - host: SSH host alias (default: "hpc.cqls_v2" or use "hpc.cqls.oregonstate.edu")
    - use_hpc_files: If True, use hpc-files.cqls.oregonstate.edu for transfer (default: False)
    
    Returns:
    - bool: True if transfer successful, False otherwise
    """
    try:
        import http.server
        import socketserver
        import threading
        import socket
        import time
        
        # Get local IP address (for WSL, get Windows host IP)
        # In WSL, Windows host is accessible via the gateway IP
        try:
            # Get default gateway (Windows host)
            result = subprocess.run(['ip', 'route', 'show', 'default'], 
                                  capture_output=True, text=True)
            if result.returncode == 0:
                gateway = result.stdout.split()[2]
                # Windows host is usually at .1 or .2 from gateway
                host_ip = gateway.rsplit('.', 1)[0] + '.1'
            else:
                host_ip = 'localhost'
        except:
            host_ip = 'localhost'
        
        # Find an available port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.bind(('', 0))
            port = s.getsockname()[1]
        
        # Get absolute paths
        local_file_abs = Path(local_file).absolute()
        file_name = local_file_abs.name
        file_dir = local_file_abs.parent
        local_size = local_file_abs.stat().st_size
        
        # Use hpc-files for large files if requested (recommended by HPC docs)
        if use_hpc_files and local_size >= 100000:
            print(f"  Using hpc-files.cqls.oregonstate.edu for large file transfer ({local_size} bytes)...")
            files_host = "hpc-files.cqls.oregonstate.edu"
            # Use scp through PowerShell
            local_path_win = str(local_file_abs)
            cmd = [
                'powershell.exe',
                '-Command',
                f'scp "{local_path_win}" {files_host}:"{remote_path}"'
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            if result.returncode == 0:
                # Verify file size
                verify_cmd = f'test -f "{remote_path}" && stat -c%s "{remote_path}"'
                verify_success, verify_stdout, _ = execute_remote_command_via_powershell(
                    verify_cmd,
                    host=host,
                    timeout=10
                )
                if verify_success and verify_stdout.strip():
                    remote_size = int(verify_stdout.strip())
                    if remote_size == local_size:
                        print("  ✓ File transferred successfully via hpc-files")
                        return True
                    else:
                        print(f"  ⚠ Size mismatch: local={local_size}, remote={remote_size}")
                else:
                    print("  ⚠ Transfer completed but verification failed")
            else:
                print(f"  ⚠ hpc-files transfer failed: {result.stderr[:200] if result.stderr else result.stdout[:200]}")
                print("  Falling back to base64/HTTP method...")
        
        # Use chunked base64 method for all files (works for both small and large files)
        # This is the most reliable method and avoids SCP/HTTP issues
        print(f"  Using chunked base64 transfer ({local_size} bytes, {local_size / 1024 / 1024:.2f} MB)...")
        
        # Create directory on remote
        remote_dir = str(Path(remote_path).parent)
        success, _, _ = execute_remote_command_via_powershell(
            f'mkdir -p {remote_dir}',
            host=host
        )
        if not success:
            print("  ⚠ Failed to create remote directory")
            return False
        
        # Read file and encode to base64
        import base64
        with open(local_file_abs, 'rb') as f:
            file_content = f.read()
        file_b64 = base64.b64encode(file_content).decode('utf-8')
        
        # Get expanded remote path (handle ~ expansion)
        success, stdout, _ = execute_remote_command_via_powershell(
            f'echo {remote_path}',
            host=host,
            timeout=10
        )
        if success and stdout.strip():
            remote_path_expanded = stdout.strip()
        else:
            remote_path_expanded = remote_path
        
        # Use Unix-style path operations (don't use Path() which may normalize to Windows paths)
        remote_path_expanded = remote_path_expanded.replace('\\\\', '/').replace('\\', '/')
        # Get directory using Unix-style string operations
        if '/' in remote_path_expanded:
            remote_dir_expanded = '/'.join(remote_path_expanded.split('/')[:-1]) if remote_path_expanded.split('/')[:-1] else '.'
        else:
            remote_dir_expanded = '.'
        remote_b64 = f"{remote_dir_expanded}/{remote_path_expanded.split('/')[-1] if '/' in remote_path_expanded else remote_path_expanded}.b64"
        
        # Split into chunks (15000 chars per chunk - safe size for PowerShell)
        chunk_size = 15000
        chunks = [file_b64[i:i+chunk_size] for i in range(0, len(file_b64), chunk_size)]
        total_chunks = len(chunks)
        
        if total_chunks > 1:
            print(f"  Writing {total_chunks} chunks (this may take a few minutes for large files)...")
        else:
            print(f"  Writing file...")
        
        # Escape function for bash double quotes
        def escape_for_bash_double_quotes(text):
            return text.replace('\\', '\\\\').replace('$', '\\$').replace('`', '\\`').replace('"', '\\"').replace('\n', '\\n')
        
        # Write first chunk
        if chunks:
            escaped_chunk = escape_for_bash_double_quotes(chunks[0])
            command = f'printf "%s" "{escaped_chunk}" > "{remote_b64}"'
            success, _, stderr = execute_remote_command_via_powershell(
                command,
                host=host,
                timeout=300
            )
            if not success:
                print(f"  ⚠ Failed to write first chunk: {stderr[:200] if stderr else 'Unknown error'}")
                print("  Falling back to HTTP method...")
            else:
                if total_chunks > 1:
                    print(f"  Progress: 1/{total_chunks} chunks...")
                
                # Append remaining chunks
                chunk_failed = False
                for i, chunk in enumerate(chunks[1:], 2):
                    if total_chunks > 20 and (i % 20 == 0 or i == total_chunks):
                        print(f"  Progress: {i}/{total_chunks} chunks...")
                    escaped_chunk = escape_for_bash_double_quotes(chunk)
                    command = f'printf "%s" "{escaped_chunk}" >> "{remote_b64}"'
                    success, _, stderr = execute_remote_command_via_powershell(
                        command,
                        host=host,
                        timeout=300
                    )
                    if not success:
                        print(f"  ⚠ Failed to append chunk {i}: {stderr[:200] if stderr else 'Unknown error'}")
                        print("  Falling back to HTTP method...")
                        chunk_failed = True
                        break
                
                if not chunk_failed:
                    # All chunks written successfully, now decode
                    print("  Decoding base64 file on remote...")
                    
                    # Create a Python script on remote to decode the file (avoids quoting issues)
                    decode_script = f"{remote_dir_expanded}/decode.py"
                    script_content = f"""import base64
import sys
import os
b64_file = '{remote_b64}'
out_file = '{remote_path_expanded}'
with open(b64_file, 'r') as f:
    data = f.read()
with open(out_file, 'wb') as f:
    f.write(base64.b64decode(data))
os.remove(b64_file)
"""
                    
                    # Write script using base64 encoding to avoid quote issues
                    script_b64 = base64.b64encode(script_content.encode('utf-8')).decode('utf-8')
                    write_script_cmd = f"echo '{script_b64}' | base64 -d > '{decode_script}'"
                    success, _, stderr = execute_remote_command_via_powershell(
                        write_script_cmd,
                        host=host,
                        timeout=60
                    )
                    if not success:
                        print(f"  ⚠ Failed to write decode script: {stderr[:200] if stderr else 'Unknown error'}")
                        print("  Falling back to HTTP method...")
                    else:
                        # Run the script
                        command = f"python3 '{decode_script}' && rm -f '{decode_script}'"
                        success, stdout, stderr = execute_remote_command_via_powershell(
                            command,
                            host=host,
                            timeout=300
                        )
                        
                        if success:
                            # Verify file size
                            verify_cmd = f'test -f "{remote_path_expanded}" && stat -c%s "{remote_path_expanded}"'
                            verify_success, verify_stdout, _ = execute_remote_command_via_powershell(
                                verify_cmd,
                                host=host,
                                timeout=10
                            )
                            if verify_success and verify_stdout.strip():
                                remote_size = int(verify_stdout.strip())
                                if remote_size == local_size:
                                    print("  ✓ File transferred successfully via chunked base64")
                                    return True
                                else:
                                    print(f"  ⚠ Size mismatch: local={local_size}, remote={remote_size}")
                                    print("  Falling back to HTTP method...")
                            else:
                                print("  ⚠ Decode completed but file verification failed")
                                print("  Falling back to HTTP method...")
                        else:
                            print(f"  ⚠ Decode failed: {stderr[:200] if stderr else stdout[:200]}")
                            print("  Falling back to HTTP method...")
        
        # Store original working directory
        original_cwd = os.getcwd()
        
        # Create a simple HTTP server in the file's directory
        os.chdir(str(file_dir))
        
        # Start HTTP server in background thread
        handler = http.server.SimpleHTTPRequestHandler
        httpd = socketserver.TCPServer(("", port), handler)
        httpd.timeout = 1
        
        def serve():
            httpd.serve_forever()
        
        server_thread = threading.Thread(target=serve, daemon=True)
        server_thread.start()
        
        # Give server a moment to start
        time.sleep(1)
        
        try:
            # Get the Windows hostname/IP that HPC can reach
            # For WSL, we need to use the Windows host IP
            # Try to get it from PowerShell
            try:
                cmd = ['powershell.exe', '-Command', 
                       '(Get-NetIPAddress -AddressFamily IPv4 | Where-Object {$_.IPAddress -like "192.168.*" -or $_.IPAddress -like "10.*"}).IPAddress | Select-Object -First 1']
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=5)
                if result.returncode == 0 and result.stdout.strip():
                    windows_ip = result.stdout.strip()
                else:
                    windows_ip = host_ip
            except:
                windows_ip = host_ip
            
            # If localhost, try to get actual IP
            if windows_ip == 'localhost' or windows_ip.startswith('127.'):
                # Try to get WSL host IP
                try:
                    result = subprocess.run(['hostname', '-I'], 
                                          capture_output=True, text=True)
                    if result.returncode == 0:
                        wsl_ip = result.stdout.split()[0]
                        # Windows host is typically .1 or .2
                        ip_parts = wsl_ip.rsplit('.', 1)
                        windows_ip = ip_parts[0] + '.1'
                except:
                    windows_ip = 'localhost'
            
            # Create directory on remote
            remote_dir = str(Path(remote_path).parent)
            success, _, _ = execute_remote_command_via_powershell(
                f'mkdir -p {remote_dir}',
                host=host
            )
            
            # Use wget to download from local HTTP server
            # Note: HPC needs to be able to reach your local machine
            # If on VPN, this should work. If not, may need to use hpc-files
            url = f"http://{windows_ip}:{port}/{file_name}"
            print(f"  Serving file at http://{windows_ip}:{port}/{file_name}")
            print(f"  Downloading to {remote_path}...")
            
            # Try wget first, fallback to curl
            command = f'wget -O "{remote_path}" "{url}" 2>&1 || curl -o "{remote_path}" "{url}" 2>&1'
            success, stdout, stderr = execute_remote_command_via_powershell(
                command,
                host=host,
                timeout=300
            )
            
            # Check if file was downloaded successfully
            # Verify by checking if remote file exists and has correct size
            if success:
                # Give it a moment, then verify file exists on remote
                time.sleep(1)
                verify_cmd = f'test -f "{remote_path}" && stat -c%s "{remote_path}"'
                verify_success, verify_stdout, _ = execute_remote_command_via_powershell(
                    verify_cmd,
                    host=host,
                    timeout=10
                )
                
                if verify_success and verify_stdout.strip():
                    remote_size = int(verify_stdout.strip())
                    local_size = local_file_abs.stat().st_size  # Use absolute path
                    if remote_size == local_size:
                        print("  ✓ File transferred successfully via HTTP")
                        return True
                    else:
                        print(f"  ⚠ Size mismatch: local={local_size}, remote={remote_size}")
                elif "saved" in stdout.lower() or "100%" in stdout:
                    print("  ✓ File transfer appears successful (wget/curl reported success)")
                    return True
            else:
                # Fallback: try using hpc-files if available
                print("  ⚠ Direct HTTP transfer failed, trying alternative...")
                print(f"  Error: {stderr[:200] if stderr else stdout[:200]}")
                
                # Alternative: use base64 echo method (for small files only)
                local_size = local_file_abs.stat().st_size  # Use absolute path
                if local_size < 100000:  # < 100KB
                    print("  Trying base64 method for small file...")
                    import base64
                    with open(local_file_abs, 'rb') as f:
                        file_content = f.read()
                    file_b64 = base64.b64encode(file_content).decode('utf-8')
                    
                    # Use echo with base64 decode
                    command = f"echo '{file_b64}' | base64 -d > '{remote_path}'"
                    success, stdout, stderr = execute_remote_command_via_powershell(
                        command,
                        host=host,
                        timeout=60
                    )
                    if success:
                        # Verify file was transferred correctly
                        verify_cmd = f'test -f "{remote_path}" && stat -c%s "{remote_path}"'
                        verify_success, verify_stdout, _ = execute_remote_command_via_powershell(
                            verify_cmd,
                            host=host,
                            timeout=10
                        )
                        if verify_success and verify_stdout.strip():
                            remote_size = int(verify_stdout.strip())
                            if remote_size == local_size:
                                print("  ✓ File transferred via base64")
                                return True
                
                return False
                
        finally:
            # Stop HTTP server
            try:
                httpd.shutdown()
                httpd.server_close()
            except:
                pass
            # Restore original working directory
            try:
                os.chdir(original_cwd)
            except:
                pass
                
    except Exception as e:
        print(f"⚠ Error transferring file: {e}")
        import traceback
        traceback.print_exc()
        return False


def transfer_file_from_hpc_via_powershell(remote_file, local_path, host="hpc.cqls_v2", use_hpc_files=False):
    """
    Transfer a file from HPC cluster using PowerShell SCP from WSL.
    Can use hpc-files.cqls.oregonstate.edu for large file transfers.
    
    Parameters:
    - remote_file: Path to file on remote server
    - local_path: Destination path on local machine
    - host: SSH host alias (default: "hpc.cqls_v2" or use "hpc.cqls.oregonstate.edu")
    - use_hpc_files: If True, use hpc-files.cqls.oregonstate.edu for transfer (default: False)
    
    Returns:
    - bool: True if transfer successful, False otherwise
    """
    try:
        local_path_win = str(Path(local_path).absolute())
        
        cmd = [
            'powershell.exe',
            '-Command',
            f'scp {host}:"{remote_file}" "{local_path_win}"'
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
        
        if result.returncode == 0:
            return True
        else:
            print(f"⚠ File download failed: {result.stderr}")
            return False
    except Exception as e:
        print(f"⚠ Error downloading file: {e}")
        return False


def execute_remote_command_via_powershell(command, host="hpc.cqls_v2", timeout=60):
    """
    Execute a command on remote HPC cluster using PowerShell SSH.
    
    Note: Official HPC hostname is hpc.cqls.oregonstate.edu
    If using SSH config alias (e.g., hpc.cqls_v2), ensure it's configured correctly.
    
    Parameters:
    - command: Command to execute on remote server
    - host: SSH host alias (default: "hpc.cqls_v2" or use "hpc.cqls.oregonstate.edu")
    - timeout: Command timeout in seconds (default: 60)
    
    Returns:
    - tuple: (success: bool, stdout: str, stderr: str)
    """
    try:
        cmd = [
            'powershell.exe',
            '-Command',
            f'ssh {host} "{command}"'
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        
        return (result.returncode == 0, result.stdout, result.stderr)
    except subprocess.TimeoutExpired:
        return (False, "", "Command timed out")
    except Exception as e:
        return (False, "", str(e))


def submit_rosetta_job_on_hpc(remote_work_dir, xml_script, combined_pdb, partners_str, 
                               nstruct=10, relax=True, host="hpc.cqls_v2", 
                               rosetta_bin_path=None, job_name="rosetta_docking"):
    """
    Submit a Rosetta docking job on HPC cluster.
    
    Note: Official HPC hostname is hpc.cqls.oregonstate.edu (or use SSH config alias like hpc.cqls_v2).
    
    Parameters:
    - remote_work_dir: Working directory on HPC
    - xml_script: Path to XML script on HPC
    - combined_pdb: Path to combined PDB on HPC
    - partners_str: Partners string (e.g., "A_B")
    - nstruct: Number of structures
    - relax: Whether to use relaxation
    - host: SSH host alias
    - rosetta_bin_path: Path to Rosetta binaries on HPC (auto-detect if None)
    - job_name: Name for the job
    
    Returns:
    - str: Job ID or None if submission failed
    """
    print(f"\nSubmitting Rosetta job to HPC: {host}")
    
    # Track if Rosetta needs module loading
    needs_module_load = False
    rosetta_module_name = None
    
    # Auto-detect Rosetta path if not provided
    if rosetta_bin_path is None:
        print("  Detecting Rosetta installation on HPC...")
        
        # Try multiple detection methods
        detection_commands = [
            # Check known installation location first (most common)
            ('test -f ~/rosetta/source/bin/rosetta_scripts.linuxgccrelease && echo ~/rosetta/source/bin/rosetta_scripts.linuxgccrelease || echo ""', False),
            # Check if in PATH
            ('which rosetta_scripts.linuxgccrelease', False),
            # Check common module locations
            ('module avail rosetta 2>&1 | head -5', True),
            # Check if loaded as module
            ('module list 2>&1 | grep -i rosetta', True),
            # Check common installation locations
            ('find ~ -name "rosetta_scripts.linuxgccrelease" -type f 2>/dev/null | head -1', False),
            ('find /opt -name "rosetta_scripts.linuxgccrelease" -type f 2>/dev/null | head -1', False),
            ('find /usr/local -name "rosetta_scripts.linuxgccrelease" -type f 2>/dev/null | head -1', False),
        ]
        
        rosetta_found = False
        for cmd, is_module_cmd in detection_commands:
            success, stdout, _ = execute_remote_command_via_powershell(
                'find ~ -path "*/rosetta/source/bin/rosetta_scripts.linuxgccrelease" -type l 2>/dev/null | head -1',
                host=host
            )
            if success and stdout.strip():
                # If it's a module command, try to load it
                if is_module_cmd and 'rosetta' in stdout.lower():
                    print(f"  Found Rosetta module, will load in job script...")
                    # Extract module name (could be rosetta, rosetta/2023, etc.)
                    lines = stdout.strip().split('\n')
                    for line in lines:
                        if 'rosetta' in line.lower() and not line.startswith('--'):
                            # Try to extract module name
                            parts = line.split()
                            if parts:
                                rosetta_module_name = parts[0]
                                break
                    if not rosetta_module_name:
                        rosetta_module_name = 'rosetta'
                    
                    # Check if module load works and get path
                    load_cmd = f'module load {rosetta_module_name} 2>&1 && which rosetta_scripts.linuxgccrelease'
                    success2, stdout2, _ = execute_remote_command_via_powershell(
                        load_cmd,
                        host=host,
                        timeout=30
                    )
                    if success2 and stdout2.strip():
                        rosetta_bin_path = Path(stdout2.strip()).parent
                        rosetta_found = True
                        needs_module_load = True
                        print(f"  ✓ Rosetta found via module '{rosetta_module_name}': {rosetta_bin_path}")
                        break
                elif 'rosetta_scripts' in stdout and not is_module_cmd:
                    # Handle both full path and just filename
                    found_path = stdout.strip()
                    if found_path.endswith('rosetta_scripts.linuxgccrelease'):
                        rosetta_bin_path = Path(found_path).parent
                    else:
                        rosetta_bin_path = Path(found_path).parent
                    rosetta_found = True
                    print(f"  ✓ Rosetta found: {rosetta_bin_path}")
                    break
        
        if not rosetta_found:
            print("  ⚠ Rosetta not found on HPC cluster")
            print("\n  Rosetta installation options on HPC:")
            print("    1. Check if Rosetta is available as a module:")
            print("       ssh hpc.cqls_v2 'module avail rosetta'")
            print("    2. Install Rosetta in your home directory:")
            print("       ssh hpc.cqls_v2 'cd ~ && git clone https://github.com/RosettaCommons/rosetta.git'")
            print("    3. Or specify rosetta_bin_path when calling the function")
            return None
    
    # Verify Rosetta binary exists
    rosetta_scripts = f"{rosetta_bin_path}/rosetta_scripts.linuxgccrelease"
    success, stdout, stderr = execute_remote_command_via_powershell(
        f'test -f "{rosetta_scripts}" && echo "EXISTS" || echo "NOT_FOUND"',
        host=host,
        timeout=10
    )
    
    if not success or "NOT_FOUND" in stdout:
        print(f"  ⚠ Rosetta binary not found at: {rosetta_scripts}")
        print(f"  Please install Rosetta or specify the correct rosetta_bin_path")
        return None
    
    print(f"  Using Rosetta: {rosetta_scripts}")
    
    # Build Rosetta command
    rosetta_cmd = [
        str(rosetta_scripts),
        "-parser:protocol", str(xml_script),
        "-s", str(combined_pdb),
        "-partners", partners_str,
        "-nstruct", str(nstruct),
        "-out:path:pdb", str(remote_work_dir),
        "-out:file:scorefile", f"{remote_work_dir}/docking_scores.sc",
        "-ex1", "-ex2",
        "-use_input_sc",
        "-flip_HNQ",
        "-no_optH", "false"
    ]
    
    if relax:
        rosetta_cmd.append("-relax:fast")
    
    # Create a job script
    # Add module loading if needed
    module_load_section = ""
    if needs_module_load and rosetta_module_name:
        module_load_section = f"""
# Load Rosetta module
module load {rosetta_module_name}
"""
    
    job_script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={remote_work_dir}/{job_name}_%j.out
#SBATCH --error={remote_work_dir}/{job_name}_%j.err
#SBATCH --time=24:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=4
{module_load_section}
cd {remote_work_dir}
{" ".join(rosetta_cmd)}
"""
    
    # Write job script to remote
    job_script_path = f"{remote_work_dir}/{job_name}.sh"
    
    # Create a temporary file locally, transfer it, then submit
    import tempfile
    with tempfile.NamedTemporaryFile(mode='w', suffix='.sh', delete=False) as tmp:
        tmp.write(job_script)
        tmp_path = tmp.name
    
    try:
        # Transfer job script
        if not transfer_file_to_hpc_via_powershell(tmp_path, job_script_path, host=host):
            return None
        
        # Make it executable
        success, _, _ = execute_remote_command_via_powershell(
            f'chmod +x {job_script_path}',
            host=host
        )
        
        # Check for SLURM availability and submit job
        # First check if sbatch is available
        check_cmd = "which sbatch >/dev/null 2>&1 && echo 'OK' || echo 'NOT_FOUND'"
        check_success, check_stdout, _ = execute_remote_command_via_powershell(
            check_cmd,
            host=host,
            timeout=10
        )
        
        slurm_available = check_success and 'OK' in check_stdout
        
        if slurm_available:
            # Submit job using sbatch
            success, stdout, stderr = execute_remote_command_via_powershell(
                f'cd {remote_work_dir} ; sbatch {job_script_path} 2>&1',
                host=host,
                timeout=30
            )
            
            # Check for successful submission
            if success and "Submitted batch job" in stdout:
                # Extract job ID
                import re
                match = re.search(r'(\d+)', stdout)
                if match:
                    job_id = match.group(1)
                    print(f"✓ Job submitted successfully! Job ID: {job_id}")
                    return job_id
            else:
                # Check for specific errors (like invalid partition)
                error_output = stdout + " " + stderr
                if "invalid partition" in error_output.lower() or "Invalid partition" in error_output:
                    print(f"⚠ SLURM partition error detected")
                    print(f"  Error: {stdout[:200]}")
                    # Try with explicit partition all.q (default on CQLS)
                    print(f"  → Retrying with partition=all.q...")
                    success2, stdout2, stderr2 = execute_remote_command_via_powershell(
                        f'cd {remote_work_dir} ; sbatch --partition=all.q {job_script_path} 2>&1',
                        host=host,
                        timeout=30
                    )
                    if success2 and "Submitted batch job" in stdout2:
                        import re
                        match = re.search(r'(\d+)', stdout2)
                        if match:
                            job_id = match.group(1)
                            print(f"✓ Job submitted successfully! Job ID: {job_id}")
                            return job_id
                print(f"⚠ SLURM submission failed: {stdout[:300]}")
        else:
            print("⚠ SLURM (sbatch) not available, running job directly in background...")
        
        # Fallback to direct execution if SLURM not available or submission failed
        print("⚠ Running job directly in background...")
        success, _, _ = execute_remote_command_via_powershell(
            f'cd {remote_work_dir} && nohup bash {job_script_path} > {remote_work_dir}/{job_name}.out 2>&1 & echo $!',
            host=host,
            timeout=30
        )
        if success:
            return "DIRECT"
        
        print(f"⚠ Job submission failed")
        return None
        return None
        
    finally:
        # Clean up temp file
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)


def check_job_status(job_id, remote_work_dir, host="hpc.cqls_v2"):
    """
    Check status of a submitted job.
    
    Parameters:
    - job_id: Job ID or "DIRECT" for direct execution
    - remote_work_dir: Working directory on HPC
    - host: SSH host alias
    
    Returns:
    - str: Job status ("RUNNING", "COMPLETED", "FAILED", "UNKNOWN")
    """
    if job_id == "DIRECT":
        # Check if output file exists and has recent activity
        success, stdout, _ = execute_remote_command_via_powershell(
            f'test -f {remote_work_dir}/docking_scores.sc && echo "COMPLETED" || echo "RUNNING"',
            host=host
        )
        if success:
            return stdout.strip()
        return "UNKNOWN"
    else:
        # Check SLURM job status
        success, stdout, _ = execute_remote_command_via_powershell(
            f'squeue -j {job_id} 2>/dev/null | grep {job_id} && echo "RUNNING" || (sacct -j {job_id} --format=State --noheader 2>/dev/null | tail -1 | grep -q COMPLETED && echo "COMPLETED" || echo "FAILED")',
            host=host
        )
        if success:
            return stdout.strip().split()[-1] if stdout.strip() else "UNKNOWN"
        return "UNKNOWN"


def find_best_rosetta_structure(output_path):
    """Find best structure from Rosetta docking results"""
    score_file = output_path / "docking_scores.sc"
    
    if not score_file.exists():
        # Look for any PDB files
        pdb_files = list(output_path.glob("*.pdb"))
        if pdb_files:
            return pdb_files[0]
        return None
    
    try:
        best_score = float('inf')
        best_structure = None
        
        with open(score_file, 'r') as f:
            for line in f:
                if line.startswith("SCORE:"):
                    parts = line.split()
                    if "total_score" in parts:
                        score_idx = parts.index("total_score")
                        if score_idx + 1 < len(parts):
                            try:
                                score = float(parts[score_idx + 1])
                                if score < best_score:
                                    best_score = score
                                    # Find corresponding PDB file
                                    if "description" in parts:
                                        desc_idx = parts.index("description")
                                        if desc_idx + 1 < len(parts):
                                            desc = parts[desc_idx + 1]
                                            pdb_file = output_path / f"{desc}.pdb"
                                            if pdb_file.exists():
                                                best_structure = pdb_file
                            except (ValueError, IndexError):
                                continue
        
        return best_structure
        
    except Exception as e:
        print(f"⚠ Error parsing score file: {e}")
        # Fallback: return first PDB file
        pdb_files = list(output_path.glob("*.pdb"))
        return pdb_files[0] if pdb_files else None


def download_hpc_results(remote_work_dir, local_output_dir, host="hpc.cqls_v2"):
    """
    Download results from HPC cluster after job completion.
    
    Parameters:
    - remote_work_dir: Working directory on HPC where results are stored
    - local_output_dir: Local directory to save results
    - host: SSH host alias
    
    Returns:
    - Path to best structure, or None if download failed
    """
    print("=" * 60)
    print("Downloading HPC Results")
    print("=" * 60)
    print(f"Remote directory: {remote_work_dir}")
    print(f"Local directory: {local_output_dir}")
    
    output_path = Path(local_output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Download score file
    print("\nDownloading score file...")
    remote_score_file = f"{remote_work_dir}/docking_scores.sc"
    local_score_file = output_path / "docking_scores.sc"
    
    if transfer_file_from_hpc_via_powershell(remote_score_file, str(local_score_file), host=host):
        print("✓ Score file downloaded")
    else:
        print("⚠ Failed to download score file")
    
    # Download PDB files
    print("\nDownloading PDB files...")
    success, stdout, _ = execute_remote_command_via_powershell(
        f'ls {remote_work_dir}/*.pdb 2>/dev/null',
        host=host
    )
    
    if success and stdout.strip():
        pdb_files = [f.strip() for f in stdout.strip().split('\n') if f.strip()]
        print(f"Found {len(pdb_files)} PDB files")
        
        for remote_pdb_file in pdb_files:
            filename = Path(remote_pdb_file).name
            local_pdb_file = output_path / filename
            if transfer_file_from_hpc_via_powershell(remote_pdb_file, str(local_pdb_file), host=host):
                print(f"  ✓ {filename}")
    
    # Find best structure
    best_structure = find_best_rosetta_structure(output_path)
    if best_structure:
        print(f"\n✓ Results downloaded successfully!")
        print(f"Best structure: {best_structure}")
        return str(best_structure)
    else:
        print("\n⚠ Results downloaded but best structure not found")
        return None


In [8]:
# Rosetta Suite Integration
# Comprehensive Rosetta tool detection and integration


def _find_rosetta_binaries():
    """
    Find Rosetta binaries from multiple possible locations:
    1. System PATH
    2. Local rosetta/source/bin directory (from install_rosetta.sh)
    3. Conda environment
    """
    rosetta_bin = {}
    rosetta_paths = []
    
    # Check system PATH
    for cmd in ["rosetta_scripts", "relax", "docking_protocol", "fixbb"]:
        result = subprocess.run(["which", cmd], capture_output=True)
        if result.returncode == 0:
            rosetta_bin[cmd] = result.stdout.decode().strip()
            rosetta_paths.append(Path(rosetta_bin[cmd]).parent)
    
    # Check local rosetta installation (from install_rosetta.sh)
    # Rosetta structure: rosetta/source/bin (binaries built here after ./install_rosetta.sh)
    rosetta_dir = Path("rosetta")
    local_rosetta_bin = Path("rosetta/source/bin")
    
    # Verify rosetta directory exists
    if rosetta_dir.exists() and rosetta_dir.is_dir():
        # Check if binaries directory exists
        if local_rosetta_bin.exists() and local_rosetta_bin.is_dir():
            # Look for actual Rosetta binaries (not just any executable)
            rosetta_binary_names = ["relax", "rosetta_scripts", "docking_protocol", "fixbb", 
                                    "rosetta_scripts.linuxgccrelease", "relax.linuxgccrelease"]
            for cmd_file in local_rosetta_bin.glob("*"):
                if cmd_file.is_file() and os.access(cmd_file, os.X_OK):
                    cmd_name = cmd_file.name
                    # Check if it's a Rosetta binary (by name or if directory has Rosetta binaries)
                    if cmd_name in rosetta_binary_names or any(rosetta_name in cmd_name for rosetta_name in ["rosetta", "relax", "docking", "fixbb"]):
                        if cmd_name not in rosetta_bin:
                            rosetta_bin[cmd_name] = str(cmd_file.absolute())
                            rosetta_paths.append(local_rosetta_bin)
                            
                            # Also add simplified key (without extension) for easier lookup
                            # e.g., "relax.linuxgccrelease" -> also add "relax"
                            if ".linuxgccrelease" in cmd_name:
                                base_name = cmd_name.replace(".linuxgccrelease", "")
                                # Remove .default if present
                                if base_name.endswith(".default"):
                                    base_name = base_name.replace(".default", "")
                                # Only add if not already present (prefer non-extension version)
                                if base_name not in rosetta_bin:
                                    rosetta_bin[base_name] = str(cmd_file.absolute())
    
    # Check conda environment
    if "CONDA_PREFIX" in os.environ:
        conda_bin = Path(os.environ["CONDA_PREFIX"]) / "bin"
        if conda_bin.exists():
            for cmd in ["rosetta_scripts", "relax"]:
                conda_cmd = conda_bin / cmd
                if conda_cmd.exists() and cmd not in rosetta_bin:
                    rosetta_bin[cmd] = str(conda_cmd)
                    rosetta_paths.append(conda_bin)
    
    return rosetta_bin, rosetta_paths

def get_rosetta_command(cmd_name):
    """Get full path to a Rosetta command"""
    # First try exact match
    if cmd_name in rosetta_commands:
        return rosetta_commands[cmd_name]
    
    # Try with .linuxgccrelease extension
    if f"{cmd_name}.linuxgccrelease" in rosetta_commands:
        return rosetta_commands[f"{cmd_name}.linuxgccrelease"]
    
    # Try with .default.linuxgccrelease extension
    if f"{cmd_name}.default.linuxgccrelease" in rosetta_commands:
        return rosetta_commands[f"{cmd_name}.default.linuxgccrelease"]
    
    # Try to find any key that starts with cmd_name
    for key in rosetta_commands.keys():
        if key.startswith(cmd_name):
            return rosetta_commands[key]
    
    return None

# Detect Rosetta installation
rosetta_commands, rosetta_paths = _find_rosetta_binaries()
ROSETTA_AVAILABLE = len(rosetta_commands) > 0

# Print status
print("=" * 60)
print("Rosetta Suite Detection")
print("=" * 60)

if ROSETTA_AVAILABLE:
    print(f"✓ Rosetta found ({len(rosetta_commands)} command(s))")
    if rosetta_paths:
        unique_paths = list(set(str(p) for p in rosetta_paths))
        print(f"  Location(s): {', '.join(unique_paths[:2])}")
    
    # Show only relevant tools for docking workflow
    relevant_tools = ["relax", "rosetta_scripts", "docking_protocol", "fixbb"]
    found_relevant = []
    
    print("\nRelevant Rosetta tools for docking workflow:")
    for tool in relevant_tools:
        if tool in rosetta_commands:
            print(f"  ✓ {tool}")
            found_relevant.append(tool)
        else:
            # Check for variants with extensions
            variants = [k for k in rosetta_commands.keys() if k.startswith(tool) and tool in k]
            if variants:
                # Prefer the one without extension, or first variant
                preferred = [v for v in variants if not v.endswith('.linuxgccrelease')]
                if preferred:
                    print(f"  ✓ {preferred[0]}")
                    found_relevant.append(tool)
                else:
                    print(f"  ✓ {variants[0]}")
                    found_relevant.append(tool)
    
    if len(found_relevant) < len(relevant_tools):
        missing = [t for t in relevant_tools if t not in found_relevant]
        if missing:
            print(f"\n  ⚠ Note: Some tools not found: {', '.join(missing)}")
    
    # Set ROSETTA_BIN_PATH for subprocess calls
    if rosetta_paths:
        ROSETTA_BIN_PATH = str(rosetta_paths[0])
        os.environ["ROSETTA_BIN_PATH"] = ROSETTA_BIN_PATH
    else:
        ROSETTA_BIN_PATH = None
else:
    print("⚠ Rosetta not found")
    print("\nInstallation options:")
    print("  1. From GitHub: ./install_rosetta.sh")
    print("  2. Via conda: conda install -c conda-forge rosetta")
    print("  3. Manual: See INSTALL_MANUAL.md")
    print("\nNote: Rosetta is required for this workflow")
    ROSETTA_BIN_PATH = None

print("=" * 60)

Rosetta Suite Detection
✓ Rosetta found (78 command(s))
  Location(s): rosetta/source/bin

Relevant Rosetta tools for docking workflow:
  ✓ relax
  ✓ rosetta_scripts
  ✓ docking_protocol
  ✓ fixbb


In [9]:
# Rosetta Peptide-Protein Docking
# Use Rosetta for advanced peptide docking against receptor

def rosetta_peptide_docking(receptor_pdb, peptide_pdb, output_dir="rosetta_docking",
                            nstruct=10, relax=True, relax_receptor=True, relax_nstruct=5,
                            use_hpc=False, hpc_host="hpc.cqls_v2",
                            hpc_work_dir=None, wait_for_completion=True, rosetta_bin_path=None, files_already_on_hpc=False):
    """
    Perform peptide-protein docking using Rosetta.
    
    Note: Official HPC hostname is hpc.cqls.oregonstate.edu
    If using SSH config alias (e.g., hpc.cqls_v2), ensure it's configured correctly.
    
    Parameters:
    - receptor_pdb: Path to receptor PDB file
    - peptide_pdb: Path to peptide PDB file
    - output_dir: Directory to save results
    - nstruct: Number of structures to generate (default: 10)
    - relax: Whether to relax structures after docking (default: True)
    - relax_receptor: Whether to relax receptor before docking (default: True)
    - relax_nstruct: Number of relaxed receptor structures to generate (default: 5)
    - use_hpc: If True, offload computation to HPC cluster (default: False)
    - hpc_host: HPC host alias for SSH (default: "hpc.cqls_v2" or use "hpc.cqls.oregonstate.edu")
    - hpc_work_dir: Working directory on HPC (default: ~/rosetta_docking_<timestamp>)
    - wait_for_completion: If True, wait for job to complete (results remain on HPC) (default: True)
    - rosetta_bin_path: Path to Rosetta binaries on HPC (default: None, auto-detect)
    - files_already_on_hpc: If True, skip file transfers and use files already on HPC (default: False)
                              When True, receptor_pdb and peptide_pdb should be HPC paths relative to hpc_work_dir
    
    Returns:
    - Path to best docked structure, or None if docking fails
    - If use_hpc=True and wait_for_completion=False, returns job_id instead
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Get Rosetta command using helper function
    rosetta_scripts = get_rosetta_command("rosetta_scripts")
    if not rosetta_scripts:
        print("⚠ rosetta_scripts not found")
        print("   Available commands:", list(rosetta_commands.keys()))
        return None
    
    print("=" * 60)
    print("Rosetta Peptide-Protein Docking")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Peptide:  {peptide_pdb}")
    print(f"Output:   {output_dir}")
    print(f"Structures: {nstruct}")
    
    # Step 1: Relax receptor if requested (only for local mode or HPC with file transfers)
    receptor_to_use = receptor_pdb
    if relax_receptor and not (use_hpc and files_already_on_hpc):
        print("\n" + "-" * 60)
        print("Step 1: Relaxing Receptor Structure")
        print("-" * 60)
        relaxed_receptor_dir = output_path / "relaxed_receptor"
        relaxed_receptor_dir.mkdir(exist_ok=True)
        
        relaxed_receptor = rosetta_relax(
            receptor_pdb,
            output_pdb=str(relaxed_receptor_dir / f"{Path(receptor_pdb).stem}_relaxed.pdb"),
            nstruct=relax_nstruct
        )
        
        if relaxed_receptor:
            # Handle case where rosetta_relax returns a directory path
            relaxed_path = Path(relaxed_receptor)
            if relaxed_path.is_dir():
                # Find best relaxed structure in the directory
                best_relaxed = find_best_rosetta_structure(relaxed_path)
                if best_relaxed:
                    receptor_to_use = str(best_relaxed)
                else:
                    # Fallback: use first PDB file found
                    pdb_files = list(relaxed_path.glob("*.pdb"))
                    if pdb_files:
                        receptor_to_use = str(pdb_files[0])
                    else:
                        print("⚠ No relaxed structures found, using original receptor")
                        receptor_to_use = receptor_pdb
            else:
                receptor_to_use = relaxed_receptor
            print(f"✓ Using relaxed receptor: {receptor_to_use}")
        else:
            print("⚠ Receptor relaxation failed, using original receptor")
            receptor_to_use = receptor_pdb
    
    # Create Rosetta XML script for peptide docking
    # Generate Rosetta XML script inline
    # Using a valid XML format for peptide-protein docking
    xml_content = """<?xml version="1.0"?>
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name="ref2015" weights="ref2015"/>
            <ScoreFunction name="ref2015_cart" weights="ref2015_cart"/>
        </SCOREFXNS>
        
        <MOVERS>
            <Docking name="docking" fullatom="1" local_refine="1" score_high="ref2015"/>
            <FastRelax name="relax" scorefxn="ref2015_cart" />
        </MOVERS>
        
        <PROTOCOLS>
            <Add mover_name="docking"/>
            <Add mover_name="relax"/>
        </PROTOCOLS>
    </ROSETTASCRIPTS>
    """
    
    # Write XML to file for Rosetta (only if not using files_already_on_hpc)
    if not (use_hpc and files_already_on_hpc):
        xml_script = output_path / "peptide_docking.xml"
        with open(xml_script, "w") as f:
            f.write(xml_content)
    
    # Step 2: Prepare input files (only if not using files_already_on_hpc)
    # When files_already_on_hpc=True, we'll combine on HPC instead
    partners_str = None
    if not (use_hpc and files_already_on_hpc):
        print("\n" + "-" * 60)
        print("Step 2: Preparing Input Complex")
        print("-" * 60)
        # Combine receptor and peptide into a single PDB for docking
        combined_pdb = output_path / "input_complex.pdb"
        receptor_chains, peptide_chain = _combine_pdb_files(receptor_to_use, peptide_pdb, combined_pdb)
        
        if not receptor_chains or not peptide_chain:
            print("⚠ Error: Could not determine chain IDs for docking partners")
            return None
        
        # Format partners string: receptor_chains_peptide_chain (e.g., "A_B" or "ABC_D")
        partners_str = f"{''.join(receptor_chains)}_{peptide_chain}"
        print(f"Docking partners: {partners_str}")
    
    # HPC Offloading
    if use_hpc:
        print("\n" + "=" * 60)
        print("HPC Offloading Mode")
        print("=" * 60)
        
        # Set up remote work directory
        if hpc_work_dir is None:
            import datetime
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            hpc_work_dir = f"~/rosetta_docking_{timestamp}"
        
        # Expand user path
        hpc_work_dir = hpc_work_dir.replace("~", "$HOME")
        # Normalize to Unix-style paths (remove Windows drive letters, backslashes)
        import re
        hpc_work_dir = re.sub(r'^[A-Za-z]:', '', hpc_work_dir)  # Remove Windows drive letters
        hpc_work_dir = hpc_work_dir.replace('\\\\', '/').replace('\\', '/')  # Convert to Unix separators
        hpc_work_dir = re.sub(r'/+', '/', hpc_work_dir)  # Normalize multiple slashes
        
        # Expand $HOME to actual Unix path (avoid Windows path issues)
        if '$HOME' in hpc_work_dir or hpc_work_dir.startswith('~'):
            success_home, stdout_home, _ = execute_remote_command_via_powershell(
                'echo `$HOME',
                host=hpc_host,
                timeout=10
            )
            if success_home and stdout_home.strip():
                home_dir = stdout_home.strip()
                # Normalize home_dir to Unix paths
                home_dir = home_dir.replace('\\\\', '/').replace('\\', '/')
                home_dir = re.sub(r'^[A-Za-z]:', '', home_dir)  # Remove Windows drive letters
                hpc_work_dir = hpc_work_dir.replace('$HOME', home_dir).replace('~', home_dir)
                # Re-normalize after expansion
                hpc_work_dir = hpc_work_dir.replace('\\\\', '/').replace('\\', '/')
                hpc_work_dir = re.sub(r'/+', '/', hpc_work_dir)
        
        # Create remote directory
        success, _, _ = execute_remote_command_via_powershell(
            f'mkdir -p {hpc_work_dir}',
            host=hpc_host
        )
        if not success:
            print(f"⚠ Failed to create remote directory: {hpc_work_dir}")
            return None
        
        print(f"Remote work directory: {hpc_work_dir}")
        
        # Handle file transfers or use existing files on HPC
        if files_already_on_hpc:
            print("\n⚠ Files already on HPC - skipping file transfers")
            print(f"  Using files from: {hpc_work_dir}")
            
            # Step 1: Relax receptor on HPC if requested
            receptor_to_use_hpc = receptor_pdb
            if relax_receptor:
                print("\n" + "-" * 60)
                print("Step 1: Relaxing Receptor Structure on HPC")
                print("-" * 60)
                relaxed_receptor_dir_hpc = f"{hpc_work_dir}/relaxed_receptor"
                
                # Create directory for relaxed receptor
                execute_remote_command_via_powershell(
                    f'mkdir -p {relaxed_receptor_dir_hpc}',
                    host=hpc_host
                )
                
                # Relax receptor on HPC using rosetta_relax
                # First, we need to get the relax command path
                if rosetta_bin_path:
                    relax_cmd_hpc = f"{rosetta_bin_path}/relax.linuxgccrelease"
                else:
                    relax_cmd_hpc = "/home/pi/kuhfeldr/rosetta/source/bin/relax.linuxgccrelease"
                
                # Check if relax command exists
                check_relax_cmd = f'test -f "{relax_cmd_hpc}" && echo "EXISTS" || echo "NOT_FOUND"'
                success_check, stdout_check, _ = execute_remote_command_via_powershell(
                    check_relax_cmd,
                    host=hpc_host,
                    timeout=10
                )
                
                if success_check and "EXISTS" in stdout_check:
                    # Build receptor path on HPC
                    receptor_pdb_clean = receptor_pdb.replace('\\', '/').replace('\\\\', '/')
                    receptor_hpc = f"{hpc_work_dir}/{receptor_pdb_clean.split('/')[-1]}" if '/' not in receptor_pdb_clean else f"{hpc_work_dir}/{receptor_pdb_clean}"
                    
                    # Handle missing .pdb extension
                    if not receptor_hpc.endswith('.pdb'):
                        check_cmd = f'test -f "{receptor_hpc}.pdb" && echo "WITH_PDB" || echo "NO_PDB"'
                        check_success, check_stdout, _ = execute_remote_command_via_powershell(check_cmd, host=hpc_host, timeout=10)
                        if check_success and "WITH_PDB" in check_stdout:
                            receptor_hpc = f"{receptor_hpc}.pdb"
                    
                    # Run relaxation on HPC
                    relaxed_output_hpc = f"{relaxed_receptor_dir_hpc}/{Path(receptor_pdb).stem}_relaxed.pdb"
                    relax_cmd = f'cd {hpc_work_dir} && {relax_cmd_hpc} -s {receptor_hpc} -nstruct {relax_nstruct} -relax:fast -out:path:pdb {relaxed_receptor_dir_hpc} -out:file:scorefile {relaxed_receptor_dir_hpc}/relax_scores.sc'
                    
                    print(f"  Running relaxation on HPC...")
                    success_relax, stdout_relax, stderr_relax = execute_remote_command_via_powershell(
                        relax_cmd,
                        host=hpc_host,
                        timeout=3600  # 1 hour timeout for relaxation
                    )
                    
                    if success_relax:
                        # Find best relaxed structure
                        find_best_cmd = f'cd {relaxed_receptor_dir_hpc} && ls -t *.pdb 2>/dev/null | head -1'
                        success_find, stdout_find, _ = execute_remote_command_via_powershell(
                            find_best_cmd,
                            host=hpc_host,
                            timeout=10
                        )
                        if success_find and stdout_find.strip():
                            receptor_to_use_hpc = stdout_find.strip()
                            print(f"✓ Using relaxed receptor: {receptor_to_use_hpc}")
                        else:
                            print("⚠ Relaxation completed but no output found, using original receptor")
                            receptor_to_use_hpc = receptor_hpc
                    else:
                        print("⚠ Receptor relaxation failed on HPC, using original receptor")
                        receptor_to_use_hpc = receptor_hpc
                else:
                    print("⚠ Relax command not found on HPC, skipping receptor relaxation")
                    receptor_to_use_hpc = receptor_pdb
            else:
                receptor_to_use_hpc = receptor_pdb
            
            # Files are already on HPC, use provided paths directly
            remote_xml = f"{hpc_work_dir}/peptide_docking.xml"
            remote_pdb = f"{hpc_work_dir}/input_complex.pdb"
            
            # Create XML script locally and transfer it (small file, transfer works reliably)
            xml_content = """<?xml version=\"1.0\"?>
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name=\"ref2015\" weights=\"ref2015\"/>
            <ScoreFunction name=\"ref2015_cart\" weights=\"ref2015_cart\"/>
        </SCOREFXNS>
        
        <MOVERS>
            <Docking name=\"docking\" fullatom=\"1\" local_refine=\"1\" score_high=\"ref2015\"/>
            <FastRelax name=\"relax\" scorefxn=\"ref2015_cart\" />
        </MOVERS>
        
        <PROTOCOLS>
            <Add mover_name=\"docking\"/>
            <Add mover_name=\"relax\"/>
        </PROTOCOLS>
    </ROSETTASCRIPTS>
    """
            # Create XML file locally temporarily
            import tempfile
            with tempfile.NamedTemporaryFile(mode='w', suffix='.xml', delete=False) as tmp_xml:
                tmp_xml.write(xml_content)
                tmp_xml_path = tmp_xml.name
            
            try:
                # Transfer XML file to HPC using the reliable transfer method
                if not transfer_file_to_hpc_via_powershell(tmp_xml_path, remote_xml, host=hpc_host):
                    print("⚠ Failed to transfer XML script to HPC")
                    return None
                print("✓ XML script transferred to HPC")
            finally:
                # Clean up temp file
                import os
                if os.path.exists(tmp_xml_path):
                    os.unlink(tmp_xml_path)
            
            # Step 2: Combine PDB files on HPC using Python
            print("\n" + "-" * 60)
            print("Step 2: Preparing Input Complex on HPC")
            print("-" * 60)
            # Use the relaxed receptor if relaxation was performed
            if relax_receptor and 'receptor_to_use_hpc' in locals():
                receptor_hpc = receptor_to_use_hpc
            else:
                # receptor_pdb and peptide_pdb are relative to hpc_work_dir
                # Use Unix-style path operations (avoid PathLib which may normalize to Windows paths)
                receptor_pdb_clean = receptor_pdb.replace('\\\\', '/').replace('\\', '/')
                receptor_hpc = f"{hpc_work_dir}/{receptor_pdb_clean.split('/')[-1]}" if '/' not in receptor_pdb_clean else f"{hpc_work_dir}/{receptor_pdb_clean}"
            
            peptide_pdb_clean = peptide_pdb.replace('\\\\', '/').replace('\\', '/')
            peptide_hpc = f"{hpc_work_dir}/{peptide_pdb_clean.split('/')[-1]}" if '/' not in peptide_pdb_clean else f"{hpc_work_dir}/{peptide_pdb_clean}"
            
            # Handle missing .pdb extension for receptor
            if not receptor_hpc.endswith('.pdb'):
                # Check if file exists with .pdb extension
                check_cmd = f'test -f "{receptor_hpc}.pdb" && echo "WITH_PDB" || test -f "{receptor_hpc}" && echo "NO_PDB" || echo "NOT_FOUND"'
                check_success, check_stdout, _ = execute_remote_command_via_powershell(check_cmd, host=hpc_host, timeout=10)
                if check_success and "WITH_PDB" in check_stdout:
                    receptor_hpc = f"{receptor_hpc}.pdb"
                elif check_success and "NOT_FOUND" in check_stdout:
                    print(f"✗ ERROR: Receptor file not found: {receptor_hpc} or {receptor_hpc}.pdb")
                    return None
            
            # Verify both files exist before combining
            verify_receptor = f'test -f "{receptor_hpc}" && echo "EXISTS" || echo "NOT_FOUND"'
            verify_peptide = f'test -f "{peptide_hpc}" && echo "EXISTS" || echo "NOT_FOUND"'
            success_r, stdout_r, _ = execute_remote_command_via_powershell(verify_receptor, host=hpc_host, timeout=10)
            success_p, stdout_p, _ = execute_remote_command_via_powershell(verify_peptide, host=hpc_host, timeout=10)
            
            if not (success_r and "EXISTS" in stdout_r):
                print(f"✗ ERROR: Receptor file not found: {receptor_hpc}")
                # List available PDB files as suggestions
                list_pdb_cmd = f'ls -1 "{hpc_work_dir}"/*.pdb 2>/dev/null | head -10'
                success_list, stdout_list, _ = execute_remote_command_via_powershell(list_pdb_cmd, host=hpc_host, timeout=10)
                if success_list and stdout_list.strip():
                    print(f"  Available PDB files in {hpc_work_dir}:")
                    for pdb_file in stdout_list.strip().split('\n'):
                        if pdb_file.strip():
                            filename = pdb_file.strip().split('/')[-1]
                            print(f"    - {filename}")
                return None
            if not (success_p and "EXISTS" in stdout_p):
                print(f"✗ ERROR: Peptide file not found: {peptide_hpc}")
                # List available files in peptide directory
                peptide_dir = '/'.join(peptide_hpc.split('/')[:-1]) if '/' in peptide_hpc else hpc_work_dir
                list_peptide_cmd = f'ls -1 "{peptide_dir}"/*.pdb 2>/dev/null | head -10'
                success_list, stdout_list, _ = execute_remote_command_via_powershell(list_peptide_cmd, host=hpc_host, timeout=10)
                if success_list and stdout_list.strip():
                    print(f"  Available PDB files in {peptide_dir}:")
                    for pdb_file in stdout_list.strip().split('\n'):
                        if pdb_file.strip():
                            filename = pdb_file.strip().split('/')[-1]
                            print(f"    - {filename}")
                return None
            
            # Use Python script on HPC to combine PDBs (avoids heredoc/SSH banner issues)
            # This version doesn't require BioPython - uses pure Python text processing
            combine_script_content = f"""import sys
import os

def get_chain_ids_from_pdb(pdb_file):
    \"\"\"Extract unique chain IDs from a PDB file\"\"\"
    chains = set()
    try:
        with open(pdb_file, 'r') as f:
            for line in f:
                if line.startswith(('ATOM', 'HETATM')):
                    if len(line) >= 21:
                        chain_id = line[21]
                        if chain_id != ' ':
                            chains.add(chain_id)
    except Exception as e:
        print(f"Warning: Could not read chain IDs from {{pdb_file}}: {{e}}", file=sys.stderr)
    return sorted(list(chains)) if chains else ['A']

def combine_pdb_files(receptor_path, peptide_path, output_path):
    \"\"\"Combine two PDB files by appending peptide atoms with new chain ID\"\"\"
    # Get chain IDs from receptor
    receptor_chains = get_chain_ids_from_pdb(receptor_path)
    
    # Determine new chain ID for peptide (next letter after receptor chains)
    if receptor_chains:
        last_chain = receptor_chains[-1]
        new_chain_id = chr(ord(last_chain) + 1) if ord(last_chain) < ord('Z') else 'A'
    else:
        new_chain_id = 'A'
    
    # Read receptor file
    receptor_lines = []
    try:
        with open(receptor_path, 'r') as f:
            receptor_lines = f.readlines()
    except Exception as e:
        print(f"ERROR: Could not read receptor file: {{e}}", file=sys.stderr)
        sys.exit(1)
    
    # Read peptide file and modify chain IDs
    peptide_lines = []
    try:
        with open(peptide_path, 'r') as f:
            for line in f:
                # Skip END/ENDMDL lines from peptide
                if line.startswith(('END', 'ENDMDL')):
                    continue
                # Update chain ID for ATOM/HETATM records
                if line.startswith(('ATOM', 'HETATM')) and len(line) >= 21:
                    # Replace chain ID
                    line = line[:21] + new_chain_id + line[22:]
                peptide_lines.append(line)
    except Exception as e:
        print(f"ERROR: Could not read peptide file: {{e}}", file=sys.stderr)
        sys.exit(1)
    
    # Write combined file
    try:
        with open(output_path, 'w') as f:
            # Write receptor lines (skip END if present)
            for line in receptor_lines:
                if not line.startswith(('END', 'ENDMDL')):
                    f.write(line)
            # Write peptide lines
            for line in peptide_lines:
                f.write(line)
            # Add END record
            f.write("END\\n")
    except Exception as e:
        print(f"ERROR: Could not write output file: {{e}}", file=sys.stderr)
        sys.exit(1)
    
    return ''.join(receptor_chains) + '_' + new_chain_id

try:
    # Verify files exist
    receptor_path = '{receptor_hpc}'
    peptide_path = '{peptide_hpc}'
    output_path = '{remote_pdb}'
    
    if not os.path.exists(receptor_path):
        print("ERROR: Receptor file not found: " + receptor_path, file=sys.stderr)
        sys.exit(1)
    if not os.path.exists(peptide_path):
        print("ERROR: Peptide file not found: " + peptide_path, file=sys.stderr)
        sys.exit(1)
    
    # Combine files
    partners_str = combine_pdb_files(receptor_path, peptide_path, output_path)
    
    # Verify output was created
    if not os.path.exists(output_path):
        print("ERROR: Failed to create output file: " + output_path, file=sys.stderr)
        sys.exit(1)
    
    print(partners_str)
except Exception as e:
    print("ERROR: " + str(e), file=sys.stderr)
    import traceback
    traceback.print_exc(file=sys.stderr)
    sys.exit(1)
"""
            
            # Create combine script locally and transfer it
            import tempfile
            combine_script_path = None
            try:
                with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as tmp_script:
                    tmp_script.write(combine_script_content)
                    combine_script_path = tmp_script.name
                
                # Transfer script to HPC
                combine_script_remote = f"{hpc_work_dir}/combine_pdb.py"
                if not transfer_file_to_hpc_via_powershell(combine_script_path, combine_script_remote, host=hpc_host):
                    print("⚠ Failed to transfer PDB combine script")
                    return None
                
                # Run the script on HPC
                run_script_cmd = f"python3 {combine_script_remote}"
                success, stdout, stderr = execute_remote_command_via_powershell(
                    run_script_cmd,
                    host=hpc_host,
                    timeout=60
                )
                
                # Clean up remote script
                execute_remote_command_via_powershell(
                    f"rm -f {combine_script_remote}",
                    host=hpc_host,
                    timeout=10
                )
            finally:
                # Clean up local temp script
                import os
                if combine_script_path and os.path.exists(combine_script_path):
                    os.unlink(combine_script_path)
            if success and stdout.strip():
                partners_str = stdout.strip().split()[-1]
            else:
                print(f"⚠ Failed to combine PDB files on HPC: {stderr[:200]}")
                partners_str = "A_B"  # Default fallback
            
            # Verify that input_complex.pdb was actually created
            verify_cmd = f'test -f "{remote_pdb}" && echo "EXISTS" || echo "NOT_FOUND"'
            verify_success, verify_stdout, _ = execute_remote_command_via_powershell(
                verify_cmd,
                host=hpc_host,
                timeout=10
            )
            
            if verify_success and "EXISTS" in verify_stdout:
                print(f"✓ Combined PDB created on HPC (partners: {partners_str})")
            else:
                print(f"✗ ERROR: input_complex.pdb was not created on HPC!")
                print(f"  Expected location: {remote_pdb}")
                print(f"  Receptor path used: {receptor_hpc}")
                print(f"  Peptide path used: {peptide_hpc}")
                
                # Show diagnostic information
                print(f"\n  Diagnostic information:")
                if not success:
                    print(f"  Combine script failed!")
                    print(f"  Stdout: {stdout[:500] if stdout else 'None'}")
                    print(f"  Stderr: {stderr[:500] if stderr else 'None'}")
                
                # Check what files actually exist
                list_files_cmd = f'ls -la "{hpc_work_dir}" | head -20'
                success_list, stdout_list, _ = execute_remote_command_via_powershell(list_files_cmd, host=hpc_host, timeout=10)
                if success_list:
                    print(f"  Files in {hpc_work_dir}:")
                    for line in stdout_list.split('\n')[:10]:
                        if line.strip():
                            print(f"    {line}")
                
                # List all available PDB files as suggestions
                list_all_pdb_cmd = f'ls -1 "{hpc_work_dir}"/*.pdb 2>/dev/null'
                success_list_all, stdout_list_all, _ = execute_remote_command_via_powershell(list_all_pdb_cmd, host=hpc_host, timeout=10)
                if success_list_all and stdout_list_all.strip():
                    print(f"  Available PDB files in {hpc_work_dir}:")
                    for pdb_file in stdout_list_all.strip().split('\n'):
                        if pdb_file.strip():
                            filename = pdb_file.strip().split('/')[-1]
                            print(f"    - {filename}")
                
                return None
        else:
            # Transfer files to HPC
            print("\nTransferring files to HPC...")
            remote_xml = f"{hpc_work_dir}/peptide_docking.xml"
            remote_pdb = f"{hpc_work_dir}/input_complex.pdb"
            
            if not transfer_file_to_hpc_via_powershell(str(xml_script), remote_xml, host=hpc_host):
                print("⚠ Failed to transfer XML script")
                return None
            print("✓ XML script transferred")
            
            if not transfer_file_to_hpc_via_powershell(str(combined_pdb), remote_pdb, host=hpc_host):
                print("⚠ Failed to transfer PDB file")
                return None
            print("✓ PDB file transferred")
        
        # Step 3: Submit job to HPC
        print("\n" + "-" * 60)
        print("Step 3: Submitting Docking Job to HPC")
        print("-" * 60)
        job_id = submit_rosetta_job_on_hpc(
            hpc_work_dir, remote_xml, remote_pdb, partners_str,
            nstruct=nstruct, relax=relax, host=hpc_host,
            rosetta_bin_path=rosetta_bin_path
        )
        
        if not job_id:
            print("⚠ Failed to submit job to HPC")
            return None
        
        if not wait_for_completion:
            print(f"\n✓ Job submitted! Job ID: {job_id}")
            print(f"Monitor with: check_job_status('{job_id}', '{hpc_work_dir}', host='{hpc_host}')")
            return job_id
        
        # Wait for job completion
        print(f"\nWaiting for job {job_id} to complete...")
        max_wait_time = 3600 * 24  # 24 hours max
        check_interval = 30  # Check every 30 seconds
        elapsed = 0
        
        while elapsed < max_wait_time:
            status = check_job_status(job_id, hpc_work_dir, host=hpc_host)
            print(f"  Status: {status} (elapsed: {elapsed//60}m)")
            
            if status == "COMPLETED":
                print("✓ Job completed!")
                break
            elif status == "FAILED":
                print("⚠ Job failed")
                return None
            
            time.sleep(check_interval)
            elapsed += check_interval
        
        if elapsed >= max_wait_time:
            print("⚠ Job timeout - results may still be available")
        
        # Download results
        # Results are available on HPC at the remote work directory
        print(f"\n✓ Job completed! Results available on HPC at: {hpc_work_dir}")
        print(f"  Score file: {hpc_work_dir}/docking_scores.sc")
        print(f"  PDB files: {hpc_work_dir}/*.pdb")
        print(f"\n  To download files manually, use:")
        print(f"    scp {hpc_host}:{hpc_work_dir}/docking_scores.sc .")
        print(f"    scp {hpc_host}:{hpc_work_dir}/*.pdb .")
        
        # Results are on HPC - return success with HPC path info
        print(f"\n✓ Docking completed successfully!")
        print(f"Results are available on HPC at: {hpc_work_dir}")
        return f"HPC:{hpc_work_dir}"
    
    # Step 3: Run Rosetta docking locally
    print("\n" + "-" * 60)
    print("Step 3: Running Docking Simulation")
    print("-" * 60)
    try:
        cmd = [
            rosetta_scripts,
            "-parser:protocol", str(xml_script),
            "-s", str(combined_pdb),
            "-partners", partners_str,  # Specify docking partners
            "-nstruct", str(nstruct),
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "docking_scores.sc"),
            "-ex1", "-ex2",  # Extra rotamer sampling
            "-use_input_sc",  # Use input side chain conformations
            "-flip_HNQ",  # Flip Asn, Gln, His
            "-no_optH", "false"  # Optimize hydrogens
        ]
        
        if relax:
            cmd.extend(["-relax:fast"])  # Fast relaxation
        
        print(f"\nRunning: {' '.join(cmd[:5])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Find best structure based on score
            best_structure = find_best_rosetta_structure(output_path)
            if best_structure:
                print(f"\n✓ Docking completed successfully!")
                print(f"Best structure: {best_structure}")
                return str(best_structure)
            else:
                print("\n⚠ Docking completed but best structure not found")
                return None
        else:
            print(f"\n⚠ Rosetta docking failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running Rosetta: {e}")
        return None


def _combine_pdb_files(receptor_pdb, peptide_pdb, output_pdb):
    """
    Combine receptor and peptide PDB files.
    
    Returns:
    - tuple: (receptor_chain_ids, peptide_chain_id) or (None, None) on error
    """
    try:
        from Bio import PDB
        
        parser = PDB.PDBParser(QUIET=True)
        receptor_structure = parser.get_structure("receptor", receptor_pdb)
        peptide_structure = parser.get_structure("peptide", peptide_pdb)
        
        # Add peptide as a new chain to receptor
        receptor_model = list(receptor_structure.get_models())[0]
        peptide_model = list(peptide_structure.get_models())[0]
        
        # Get receptor chain IDs
        receptor_chains = [chain.id for chain in receptor_model]
        
        # Get next chain ID for peptide
        new_chain_id = chr(ord('A') + len(receptor_chains))
        
        # Create new chain for peptide
        new_chain = PDB.Chain.Chain(new_chain_id)
        for chain in peptide_model:
            for residue in chain:
                new_chain.add(residue.copy())
        
        receptor_model.add(new_chain)
        
        # Save combined structure
        io = PDB.PDBIO()
        io.set_structure(receptor_structure)
        io.save(str(output_pdb))
        
        return (receptor_chains, new_chain_id)
    except Exception as e:
        print(f"⚠ Error combining PDB files: {e}")
        return (None, None)


def rosetta_relax(pdb_file, output_pdb=None, nstruct=5):
    """
    Relax/refine structure using Rosetta relax application.
    This is simpler than full docking and useful for structure refinement.
    
    Parameters:
    - pdb_file: Path to input PDB file
    - output_pdb: Path to output PDB file (default: auto-generated)
    - nstruct: Number of relaxed structures to generate (default: 5)
    
    Returns:
    - Path to best relaxed structure, or None if relaxation fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    relax_cmd = get_rosetta_command("relax")
    if not relax_cmd:
        print("⚠ Rosetta 'relax' command not found")
        print("   Available commands:", list(rosetta_commands.keys()))
        return None
    
    if output_pdb is None:
        output_pdb = Path(pdb_file).stem + "_relaxed.pdb"
    
    output_path = Path(output_pdb).parent
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta Structure Relaxation")
    print("=" * 60)
    print(f"Input:  {pdb_file}")
    print(f"Output: {output_pdb}")
    print(f"Structures: {nstruct}")
    
    try:
        cmd = [
            relax_cmd,
            "-s", str(pdb_file),
            "-nstruct", str(nstruct),
            "-relax:fast",
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "relax_scores.sc")
        ]
        
        print(f"\nRunning: {' '.join(cmd[:3])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Find best structure
            score_file = output_path / "relax_scores.sc"
            if score_file.exists():
                best_structure = find_best_rosetta_structure(output_path)
                if best_structure:
                    print(f"\n✓ Relaxation completed!")
                    print(f"Best structure: {best_structure}")
                    return str(best_structure)
            
            print("\n✓ Relaxation completed (check output directory)")
            return str(output_path)
        else:
            print(f"\n⚠ Rosetta relax failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running Rosetta relax: {e}")
        return None


def rosetta_docking_protocol(receptor_pdb, ligand_pdb, output_dir="rosetta_docking_protocol",
                             nstruct=10, docking_method="local_refine"):
    """
    Use Rosetta docking_protocol for protein-protein/peptide-protein docking.
    This is more robust than rosetta_scripts for docking.
    
    Parameters:
    - receptor_pdb: Path to receptor PDB file
    - ligand_pdb: Path to ligand/peptide PDB file
    - output_dir: Directory to save results
    - nstruct: Number of structures to generate
    - docking_method: "local_refine" (default) or "perturb"
    
    Returns:
    - Path to best docked structure, or None if docking fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available. Install Rosetta first.")
        return None
    
    docking_protocol = get_rosetta_command("docking_protocol")
    if not docking_protocol:
        print("⚠ docking_protocol not found")
        print("   Falling back to rosetta_scripts...")
        return rosetta_peptide_docking(receptor_pdb, ligand_pdb, output_dir, nstruct)
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta Docking Protocol")
    print("=" * 60)
    print(f"Receptor: {receptor_pdb}")
    print(f"Ligand:   {ligand_pdb}")
    print(f"Method:   {docking_method}")
    print(f"Structures: {nstruct}")
    
    try:
        cmd = [
            docking_protocol,
            "-s", str(receptor_pdb), str(ligand_pdb),
            "-nstruct", str(nstruct),
            "-docking", docking_method,
            "-out:path:pdb", str(output_path),
            "-out:file:scorefile", str(output_path / "docking_scores.sc")
        ]
        
        print(f"\nRunning: {' '.join(cmd[:5])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            best_structure = find_best_rosetta_structure(output_path)
            if best_structure:
                print(f"\n✓ Docking completed!")
                print(f"Best structure: {best_structure}")
                return str(best_structure)
            else:
                print("\n✓ Docking completed (check output directory)")
                return str(output_path)
        else:
            print(f"\n⚠ Docking failed:")
            print(result.stderr[:500])
            return None
            
    except Exception as e:
        print(f"\n⚠ Error running docking_protocol: {e}")
        return None


def rosetta_fixbb(pdb_file, output_pdb=None, resfile=None):
    """
    Use Rosetta fixbb (fix backbone) to redesign side chains.
    Useful for structure optimization.
    
    Parameters:
    - pdb_file: Path to input PDB file
    - output_pdb: Path to output PDB file
    - resfile: Optional resfile for specific residue design
    
    Returns:
    - Path to output structure, or None if fails
    """
    if not ROSETTA_AVAILABLE:
        print("⚠ Rosetta not available")
        return None
    
    fixbb_cmd = get_rosetta_command("fixbb")
    if not fixbb_cmd:
        print("⚠ fixbb not found")
        return None
    
    if output_pdb is None:
        output_pdb = Path(pdb_file).stem + "_fixbb.pdb"
    
    output_path = Path(output_pdb).parent
    output_path.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("Rosetta FixBB (Side Chain Design)")
    print("=" * 60)
    print(f"Input:  {pdb_file}")
    print(f"Output: {output_pdb}")
    
    try:
        cmd = [fixbb_cmd, "-s", str(pdb_file), "-out:path:pdb", str(output_path)]
        if resfile:
            cmd.extend(["-resfile", str(resfile)])
        
        print(f"\nRunning: {' '.join(cmd[:3])}...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"\n✓ FixBB completed!")
            return str(output_pdb)
        else:
            print(f"\n⚠ FixBB failed: {result.stderr[:300]}")
            return None
            
    except Exception as e:
        print(f"\n⚠ Error: {e}")
        return None


# Print Rosetta status and usage
print("\n" + "=" * 60)
print("Rosetta Functions Available")
print("=" * 60)

if ROSETTA_AVAILABLE:
    print("\n✓ Rosetta is ready!")
    print("\nAvailable functions:")
    print("1. rosetta_relax() - Structure relaxation/refinement")
    print("2. rosetta_peptide_docking() - Peptide-protein docking (rosetta_scripts)")
    print("3. rosetta_docking_protocol() - Docking using docking_protocol")
    print("4. rosetta_fixbb() - Side chain redesign")
    print("\nExample usage:")
    print("  # Relax structure")
    print("  relaxed = rosetta_relax('3fxi.pdb', nstruct=5)")
    print("  ")
    print("  # Dock peptide to receptor (with receptor relaxation)")
    print("  docked = rosetta_peptide_docking('3fxi.pdb', 'peptide.pdb', nstruct=10, relax_receptor=True)")
    print("  ")
    print("  # Dock without receptor relaxation")
    print("  docked = rosetta_peptide_docking('3fxi.pdb', 'peptide.pdb', nstruct=10, relax_receptor=False)")
else:
    print("\n⚠ Rosetta not installed")
    print("\nTo install Rosetta:")
    print("  Option 1: ./install_rosetta.sh (from GitHub)")
    print("  Option 2: conda install -c conda-forge rosetta")
    print("  Option 3: See INSTALL_MANUAL.md")
    print("\nNote: Rosetta is required for this workflow")

print("=" * 60)



Rosetta Functions Available

✓ Rosetta is ready!

Available functions:
1. rosetta_relax() - Structure relaxation/refinement
2. rosetta_peptide_docking() - Peptide-protein docking (rosetta_scripts)
3. rosetta_docking_protocol() - Docking using docking_protocol
4. rosetta_fixbb() - Side chain redesign

Example usage:
  # Relax structure
  relaxed = rosetta_relax('3fxi.pdb', nstruct=5)
  
  # Dock peptide to receptor (with receptor relaxation)
  docked = rosetta_peptide_docking('3fxi.pdb', 'peptide.pdb', nstruct=10, relax_receptor=True)
  
  # Dock without receptor relaxation
  docked = rosetta_peptide_docking('3fxi.pdb', 'peptide.pdb', nstruct=10, relax_receptor=False)


In [10]:
# Example: Run docking on HPC cluster
# Uncomment and modify as needed

# Option 1: Use files already on HPC (RECOMMENDED - no file transfers!)
# Files should be in ~/docking folder on HPC:
#   - 3fxi_0001.pdb (receptor PDB file) - use actual filename on HPC
#   - alphafold_predictions/ (folder with peptide predictions)
docked = rosetta_peptide_docking(
    '3fxi_0001.pdb',  # HPC path relative to hpc_work_dir (use actual filename on HPC)
    'alphafold_predictions/GLAPYKLRPVAA.pdb',  # HPC path relative to hpc_work_dir
    nstruct=10,
    use_hpc=True,
    hpc_host="hpc.cqls_v2",
    hpc_work_dir="~/docking",  # Use docking folder
    rosetta_bin_path="/home/pi/kuhfeldr/rosetta/source/bin",
    files_already_on_hpc=True,  # Skip file transfers!
    wait_for_completion=True
)

# 
# # Later, check status:
# # status = check_job_status(job_info, "~/my_docking_job", host="hpc.cqls_v2")
# # 
# # When complete, download results:
# # result = download_hpc_results("~/my_docking_job", "rosetta_docking", host="hpc.cqls_v2")


Rosetta Peptide-Protein Docking
Receptor: 3fxi_0001.pdb
Peptide:  alphafold_predictions/GLAPYKLRPVAA.pdb
Output:   rosetta_docking
Structures: 10

HPC Offloading Mode
⚠ Failed to create remote directory: $HOME/docking
